In [1]:
import numpy as np, pandas as pd, os
from tqdm import tqdm
from PIL import Image
from sklearn.neighbors import NearestNeighbors

from transformers import AutoProcessor, AutoModel
import torch

/home/arianarocha40/Projects/facial-recognition/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Read in the data
I pulled this data from InsightFace's open sourced data at [LinkedIn page](https://github.com/deepinsight/insightface/tree/master/recognition/_evaluation_/ijb)

In [2]:
# Read in the labels of the data

# Define the file path
file_path = "ijb/IJBB/meta/ijbb_template_pair_label.txt"

# Read the file into a pandas DataFrame
df = pd.read_csv(
    file_path,
    sep="\s+",  # Use whitespace as the delimiter
    header=None,  # No header row in the file
    names=["person_id", "image_id", "match"],  # Assign column names
    dtype={"person_id": int, "image_id": str, "match": int}  # Ensure correct data types
)

# Display the first few rows of the DataFrame
print(df.head())

   person_id image_id  match
0          1    11065      1
1          1    11066      1
2          1    11067      1
3          1    11068      1
4          1    11069      1


In [3]:
image_directory = "ijb/IJBB/loose_crop"

# Count the number of image files in the directory
image_count = len([file for file in os.listdir(image_directory) if file.endswith(".jpg")])

print(f"Number of images in the directory: {image_count}")

Number of images in the directory: 227630


# Embed the Images and Store those Embeddings

Embed images by standard CLIP

In [2]:
class clip_embed:

    def __init__(self, model_name = "google/siglip-so400m-patch14-384", device="gpu:0"):
        self.device = torch.device(device)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.processor = AutoProcessor.from_pretrained(model_name)

    def run(self, images):
        inputs = self.processor(images=images, return_tensors="pt").to(self.device)
        with torch.no_grad():
            base_embeddings = self.model.get_image_features(**inputs)
        return base_embeddings.cpu().tolist()

In [3]:
# stand up the embedding model

clip_embedder = clip_embed(device="cuda:0")


In [11]:
model = AutoModel.from_pretrained("openai/clip-vit-large-patch14")

In [23]:
model

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e

In [7]:
image_files = [f for f in os.listdir(image_directory) if f.endswith(".jpg")]
results = []

# Batch size
batch_size = 32

# Process images in batches with tqdm for progress tracking
for i in tqdm(range(0, len(image_files), batch_size), desc="Embedding images"):
    batch_files = image_files[i:i + batch_size]
    batch_images = []
    batch_image_ids = []

    # Load and prepare images for the batch
    for image_file in batch_files:
        image_id = image_file[:-4]  # Remove ".jpg" to get the image_id
        image_path = os.path.join(image_directory, image_file)
        image = Image.open(image_path).convert("RGB")  # Ensure image is in RGB format
        batch_images.append(image)
        batch_image_ids.append(image_id)

    # Get embeddings for the batch
    batch_embeddings = clip_embedder.run(batch_images)

    # Append results for the batch
    results.extend(
        {"image_id": image_id, "clip_embedding": embedding}
        for image_id, embedding in zip(batch_image_ids, batch_embeddings)
    )

Embedding images: 100%|██████████| 7114/7114 [4:44:05<00:00,  2.40s/it]  


In [8]:
# Create the results DataFrame
results_df = pd.DataFrame(results)

# Display the first few rows of the DataFrame
print(results_df.head())

  image_id                                     clip_embedding
0    74425  [0.48184943199157715, 0.39835166931152344, 0.1...
1    75130  [0.22251923382282257, 0.17251801490783691, -0....
2    61107  [0.49914735555648804, 0.2005760669708252, -0.4...
3   121375  [0.812370240688324, 0.04146367311477661, 0.131...
4    99703  [-0.1284291297197342, 0.5022444128990173, 0.59...


In [7]:
results_df.shape

(227630, 4)

In [11]:
# Save out the interim results
results_df.to_pickle("zero_shot_ijb_embeddings.pkl")

In [6]:
# Read in previously embedded results

results_df = pd.read_pickle("zero_shot_ijb_embeddings.pkl")

# Evaluate Results

Compute the 1N accuracy for the samples
- get the nearest neighbor for each image (other than itself)
- compare these neighbors to the known results to see if that neighbor is an image of the same person
- compute the overall accuracy

In [18]:
embeddings = np.array(results_df["clip_embedding"].tolist())

# Initialize NearestNeighbors with cosine similarity
nn_model = NearestNeighbors(n_neighbors=2, metric="cosine", n_jobs=-1)  # n_neighbors=2 includes the point itself
nn_model.fit(embeddings)

NearestNeighbors(metric='cosine', n_jobs=-1, n_neighbors=2)

In [19]:
# Find the nearest neighbors
distances, indices = nn_model.kneighbors(embeddings)

# Add nearest neighbor info to the DataFrame
results_df["nn"] = indices[:, 1]  # Second column corresponds to the nearest neighbor excluding itself
results_df["nn"] = results_df["nn"].apply(lambda idx: results_df.loc[idx, "image_id"])

In [20]:
# Save out the interim results with nearest neighbor
results_df.to_pickle("zero_shot_ijb_embeddings.pkl")

In [8]:
results_df.head()

,image_id,clip_embedding,nn,nn_accuracy
0,74425,"[0.48184943199157715, 0.39835166931152344, 0.1...",70903,NaN
1,75130,"[0.22251923382282257, 0.17251801490783691, -0....",75185,NaN
2,61107,"[0.49914735555648804, 0.2005760669708252, -0.4...",61107,NaN
3,121375,"[0.812370240688324, 0.04146367311477661, 0.131...",121220,NaN
4,99703,"[-0.1284291297197342, 0.5022444128990173, 0.59...",99703,NaN


In [13]:
df.head()

,person_id,image_id,match
0,1,11065,1
1,1,11066,1
2,1,11067,1
3,1,11068,1
4,1,11069,1


In [31]:
filtered_df = df[df["match"] == 1]

# Perform a left merge on "image_id"
merged_results_df = results_df.merge(filtered_df[["image_id", "person_id"]], on="image_id", how="left")
# Perform a left merge on "image_id"
merged_results_df = merged_results_df.merge(filtered_df[["image_id", "person_id"]], left_on="nn", right_on="image_id", how="left")
merged_results_df = merged_results_df[~merged_results_df["person_id_x"].isna()]

merged_results_df['nn_accuracy'] = (merged_results_df["person_id_x"] == merged_results_df["person_id_y"])

In [32]:
print(merged_results_df.head())

    image_id_x                                     clip_embedding     nn  \
25       16389  [0.3545876741409302, 0.19320034980773926, -0.1...  16298   
79       12688  [-0.2849375009536743, 0.6656880974769592, -0.2...  12528   
90       13346  [0.27334100008010864, 0.1216580867767334, -0.5...  13415   
106      11812  [-0.23596921563148499, 0.3980366587638855, -0....  12001   
124      19024  [0.2042960226535797, 0.20593667030334473, -0.1...  18303   

     nn_accuracy  person_id_x image_id_y  person_id_y  
25         False        953.0      16298        931.0  
79         False        276.0      12528        249.0  
90         False        392.0      13415        402.0  
106        False        128.0      12001        161.0  
124        False       1403.0      18303       1277.0  


In [33]:
merged_results_df['nn_accuracy'].sum() / len(merged_results_df['nn_accuracy'])

np.float64(0.27585199610516065)

In [38]:
# Add accuracy results as a column to results_df
results_df["nn_accuracy"] = accuracy_results

# Calculate the overall accuracy (ignoring NaN values)
overall_accuracy = results_df["nn_accuracy"].mean(skipna=True)
print(f"Nearest Neighbor Accuracy: {overall_accuracy:.2%}")

Nearest Neighbor Accuracy: 0.03%


In [41]:
np.unique(accuracy_results, return_counts=True)

(array([ 0.,  1., nan]), array([ 10267,      3, 217360]))

In [39]:
print(results_df.head())

  image_id                                     clip_embedding      nn  \
0    74425  [0.48184943199157715, 0.39835166931152344, 0.1...   70903   
1    75130  [0.22251923382282257, 0.17251801490783691, -0....   75185   
2    61107  [0.49914735555648804, 0.2005760669708252, -0.4...   61107   
3   121375  [0.812370240688324, 0.04146367311477661, 0.131...  121220   
4    99703  [-0.1284291297197342, 0.5022444128990173, 0.59...   99703   

   nn_accuracy  
0          NaN  
1          NaN  
2          NaN  
3          NaN  
4          NaN  


In [40]:
# Save out the interim results with evaluation
results_df.to_pickle("zero_shot_ijb_embeddings.pkl")